In [7]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import classification_report

from tensorflow.keras.layers import Input, Dense, ReLU
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

import joblib
import hls4ml

In [8]:
df = pd.read_csv("advitam_exp4_eda_hr_fatigue.csv")

print(df.shape)
df.head()

(966, 170)


,EDA_filtered_mean_Dr,EDA_filtered_min_Dr,EDA_filtered_max_Dr,EDA_filtered_std_Dr,EDA_tonic_mean_Dr,EDA_tonic_min_Dr,EDA_tonic_max_Dr,EDA_tonic_std_Dr,SCR_Peaks_freq_Dr,ECG_Rate_Mean_Dr,...,HRV_CD_Dr-Bl,HRV_HFD_Dr-Bl,HRV_KFD_Dr-Bl,HRV_LZC_Dr-Bl,SCR_Peaks_N_Dr-Bl,SCR_Peaks_Amplitude_Mean_Dr-Bl,subject_id,KSS,fatigue_class,fatigue_binary
0,8.773517,8.199914,9.790087,0.359757,8.773074,8.320003,9.586515,0.313288,6.000000,76.496241,...,-0.113947,-0.180401,-1.156949,0.021260,9.0,0.155982,1,3,Alert,Alert
1,8.000926,7.578529,9.075803,0.286905,8.001030,7.576932,8.615724,0.258925,3.000000,75.662830,...,-0.143630,-0.209031,-0.159025,0.096659,0.0,0.111683,1,3,Alert,Alert
2,7.248483,6.603319,7.723252,0.328557,7.249025,6.643583,7.689707,0.327241,4.666667,72.562112,...,-0.004692,-0.184244,-0.416431,0.199648,5.0,-0.117875,1,3,Alert,Alert
3,6.860160,6.454878,7.567762,0.283449,6.859642,6.474646,7.348672,0.269346,1.333333,73.424872,...,0.007717,-0.189353,-0.114486,0.049923,-5.0,0.133715,1,3,Alert,Alert
4,7.417586,6.808223,8.585298,0.386247,7.418061,6.833763,8.173627,0.361820,2.000000,74.200229,...,-0.036278,-0.238525,-0.742359,0.078614,-3.0,0.186818,1,3,Alert,Alert


In [9]:
X = df.drop(
    columns=[
        "subject_id",
        "fatigue_binary",
        "fatigue_class"
    ]
)

y = df["fatigue_class"]

encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [10]:
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
scaler = StandardScaler()

X_train_full = scaler.fit_transform(X_train_full)
X_test_full = scaler.transform(X_test_full)

In [12]:
selector = SelectKBest(
    score_func=mutual_info_classif,
    k=10
)

X_train = selector.fit_transform(X_train_full, y_train)
X_test = selector.transform(X_test_full)

In [13]:
y_train_cat = to_categorical(y_train, 3)
y_test_cat = to_categorical(y_test, 3)

In [14]:
inputs = Input(shape=(10,))

x = Dense(
    16,
    use_bias=False,
    kernel_regularizer=l2(1e-4)
)(inputs)
x = ReLU()(x)

x = Dense(
    8,
    use_bias=False,
    kernel_regularizer=l2(1e-4)
)(x)
x = ReLU()(x)

outputs = Dense(
    3,
    activation="softmax"
)(x)

model = Model(inputs, outputs)

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [16]:
history = model.fit(
    X_train,
    y_train_cat,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True
        )
    ],
    verbose=1
)

Epoch 1/150


20/20 [==============================] - 2s 16ms/step - loss: 1.1510 - accuracy: 0.3485 - val_loss: 1.1444 - val_accuracy: 0.3226
Epoch 2/150
20/20 [==============================] - 0s 3ms/step - loss: 1.0838 - accuracy: 0.4230 - val_loss: 1.0968 - val_accuracy: 0.3742
Epoch 3/150
20/20 [==============================] - 0s 2ms/step - loss: 1.0380 - accuracy: 0.4554 - val_loss: 1.0567 - val_accuracy: 0.4000
Epoch 4/150
20/20 [==============================] - 0s 3ms/step - loss: 0.9971 - accuracy: 0.4911 - val_loss: 1.0130 - val_accuracy: 0.4387
Epoch 5/150
20/20 [==============================] - 0s 2ms/step - loss: 0.9534 - accuracy: 0.5543 - val_loss: 0.9651 - val_accuracy: 0.5355
Epoch 6/150
20/20 [==============================] - 0s 4ms/step - loss: 0.9005 - accuracy: 0.6791 - val_loss: 0.9055 - val_accuracy: 0.6903
Epoch 7/150
20/20 [==============================] - 0s 3ms/step - loss: 0.8390 - accuracy: 0.7488 - val_loss: 0.8422 - val_accuracy: 0.7290
Epoch 8/15

In [17]:
pred = model.predict(X_test)

pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))

7/7 [==============================] - 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        62
           1       0.98      1.00      0.99        62
           2       1.00      1.00      1.00        70

    accuracy                           0.99       194
   macro avg       0.99      0.99      0.99       194
weighted avg       0.99      0.99      0.99       194



In [18]:
joblib.dump(scaler, "scaler.pkl")
joblib.dump(selector, "feature_selector.pkl")

['feature_selector.pkl']

In [19]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 10)]              0         
                                                                 
 dense (Dense)               (None, 16)                160       
                                                                 
 re_lu (ReLU)                (None, 16)                0         
                                                                 
 dense_1 (Dense)             (None, 8)                 128       
                                                                 
 re_lu_1 (ReLU)              (None, 8)                 0         
                                                                 
 dense_2 (Dense)             (None, 3)                 27        
                                                                 
Total params: 315 (1.23 KB)
Trainable params: 315 (1.23 KB)
N

In [20]:
config = hls4ml.utils.config_from_keras_model(
    model,
    granularity="name"
)

config["Model"]["Precision"] = "ap_fixed<16,6>"
config["Model"]["ReuseFactor"] = 1
config["Model"]["Strategy"] = "Resource"

print(config)

{'Model': {'Precision': 'ap_fixed<16,6>', 'ReuseFactor': 1, 'Strategy': 'Resource', 'BramFactor': 1000000000, 'TraceOutput': False}, 'LayerName': {'input_1': {'Trace': False, 'Precision': {'result': 'auto'}}, 'dense': {'Trace': False, 'Precision': {'result': 'auto', 'weight': 'auto', 'bias': 'auto'}}, 'dense_linear': {'Trace': False, 'Precision': {'result': 'auto'}}, 're_lu': {'Trace': False, 'Precision': {'result': 'auto'}}, 'dense_1': {'Trace': False, 'Precision': {'result': 'auto', 'weight': 'auto', 'bias': 'auto'}}, 'dense_1_linear': {'Trace': False, 'Precision': {'result': 'auto'}}, 're_lu_1': {'Trace': False, 'Precision': {'result': 'auto'}}, 'dense_2': {'Trace': False, 'Precision': {'result': 'auto', 'weight': 'auto', 'bias': 'auto'}}, 'dense_2_softmax': {'Trace': False, 'Precision': {'result': 'auto'}}}}


In [21]:
hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir="hls4ml_prj",
    backend="Vitis",
    part="xc7z020clg484-1"
)

In [25]:
config = hls4ml.utils.config_from_keras_model(
    model,
    granularity="name"
)

config["Model"]["Precision"] = "ap_fixed<16,6>"
config["Model"]["ReuseFactor"] = 1
config["Model"]["Strategy"] = "Resource"

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir="hls4ml_prj",
    backend="Vivado",
    part="xc7z020clg484-1"
)

In [26]:
loss, accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"Test Accuracy: {accuracy*100:.2f}%")

Test Accuracy: 99.48%
